# 📊 Data Visualization in Databricks — Project Showcase

![Databricks](https://img.shields.io/badge/Platform-Databricks-FF3621?logo=databricks&logoColor=white)
![Charts](https://img.shields.io/badge/Focus-Charts%20%26%20Dashboards-orange)
![Dataset](https://img.shields.io/badge/Dataset-NYC%20Taxi%20%7C%20Retail%20Sales-blueviolet)
![Status](https://img.shields.io/badge/Status-Completed-brightgreen)

**Scenario I worked through:** acting as a data analyst for a taxi company (using the public **NYC Taxi Trips** dataset) and, for the mapping exercises, a global retail company (using an **online retail** sales dataset) — building the full range of chart types Databricks SQL supports, then combining them into dashboards that track operations, are refreshed on a schedule, and are shared securely with different audiences.

**What this notebook demonstrates:** the ability to choose the right chart for a question, apply formatting that makes a chart trustworthy and easy to read, and manage dashboards end-to-end — creating, editing, publishing, scheduling, cloning, exporting, sharing, and alerting.

---


## Chapter 1 — Core Chart Types

I started with the fundamentals: bar, line, combo, and map visualizations, each built to answer a specific question about NYC taxi fares or global retail sales.


### 1.1 — Bar chart: average fare by hour of day

```sql
SELECT *, hour(tpep_pickup_datetime) AS pickup_hour, month(tpep_pickup_datetime) AS month
FROM samples.nyctaxi.trips;
```
Bar visualization: X = `pickup_hour`, Y = average of `fare_amount`.

**Result: 5 AM** has the highest average fare — a useful early signal that off-peak trips (likely longer airport/outer-borough runs) command a premium.

![Average fare by hour — bar chart](./screenshots/image1.png)


### 1.2 — Line chart: fare stability by hour

Same base query, this time visualized as a line chart of the **standard deviation** of `fare_amount` by hour, to find the *most predictable* pricing window rather than just the highest average.

**Result: 7 PM** has the lowest standard deviation — the most consistent, predictable fare window of the day.

![Standard deviation of fare by hour — line chart](./screenshots/image2.png)


### 1.3 — Combo chart: volume and revenue together

A **combo chart** (bar + line in one view) let me compare `sum(trip_distance)` and `sum(fare_amount)` by hour simultaneously, instead of needing two separate charts to see how ride volume and revenue move together.

**Result: 5 AM** has both the lowest total fare amount *and* the lowest total trip distance — consistent with it simply being the lowest-volume hour, which reframes the "highest average fare" finding from 1.1: it's a high average over very few trips, not a high-revenue hour.

![Combo chart: trip distance and fare amount by hour](./screenshots/image3.png)


### 1.4 — Choropleth map: sales by country

Switching to the retail dataset, I built a **choropleth map** shading each country by total sales value — the right chart when the question is fundamentally geographic ("where are we strong, where are we weak?").

```sql
SELECT Country, Latitude, Longitude,
       COUNT(DISTINCT CustomerID) AS UniqueCustomerCount,
       SUM(Quantity * UnitPrice)  AS TotalPrice
FROM online_retail
GROUP BY Country, Latitude, Longitude
ORDER BY Country;
```

**Result:** the **United Kingdom** has the highest total sales by a wide margin — expected for a UK-based retailer, and a useful sanity check before digging into international markets.

![Choropleth map of total sales by country](./screenshots/image4.png)


### 1.5 — Marker map: customer distribution by location

Using the same query, a **marker map** plots individual geographic points rather than shading whole regions — better suited to answering a location-specific question like customer counts in a single country.

**Result:** **France has 30 unique customers** in the dataset.

![Marker map of customer distribution](./screenshots/image5.png)


### 1.6 — Finding the most profitable day of the week

I engineered a `day_of_week` field from `tpep_pickup_datetime` with a `CASE` statement and built a bar plot of total fare by day, filtering trips to a realistic range (`0 < trip_distance < 10`, `0 < fare_amount < 50`) to avoid outliers skewing the comparison.

![Total fare amount by day of week](./screenshots/image6.png)

---


## Chapter 2 — Formatting, Storytelling & Customizable Tables

A correct chart that's hard to read is still a bad chart. This chapter was about **formatting discipline**: clear axis labels, deliberate color choices (not "rainbow" palettes), meaningful data labels, and tables customized to highlight what actually matters to the reader.


### 2.1 — Configuring axis titles and series labels

Working with two months of fare data, I relabeled the series as "Jan" and "Feb," renamed the Y-axis to "Total Fare Amount," and added data labels — small changes that make the difference between a chart that needs a caption and one that's self-explanatory.

**Result:** total fare amount for **February was $136,671.67**.

![Bar chart with configured axis titles and series labels](./screenshots/image7.png)


### 2.2 — Color coding and numeric label formatting

On an hourly, two-month line chart, I set February's line to red for immediate visual contrast, renamed the Y-axis, and formatted the data labels to abbreviate large numbers (`"0a"` format, e.g. `9k` instead of `9000`) — a small formatting choice that keeps a busy chart readable.

**Result:** the maximum labeled value on the red (February) line is **9k**.

![Line chart with custom color and abbreviated data labels](./screenshots/image8.png)


### 2.3 — Pie chart with precise percentage labels

For a part-to-whole question — how each hour contributes to total daily fares — I built a pie chart and formatted the data labels to one decimal place for precision.

**Result: 7 PM** is the second-highest slice, representing **5.9%** of total fare amount.

![Pie chart of fare distribution by hour](./screenshots/image9.png)


### 2.4 — Histogram with adjusted bin granularity

To understand the distribution of `trip_distance`, I built a histogram and increased the bin count to 50 for a more granular view, with clearly labeled axes ("Trip Distance (miles)" / "Count of Trips").

**Result:** the 1.836–2.448 mile bin — the third-highest bar — contains **2,658 trips**.

![Histogram of trip distance with 50 bins](./screenshots/image10.png)


### 2.5 — Highlighting a single data point with strategic color

For an "evening trip counts" bar chart (hours ≥ 17:00), I set the tallest bar to red and all others to blue, immediately drawing the eye to the peak hour without needing a separate annotation — a simple but effective storytelling technique.

![Evening trip counts with the peak hour highlighted in red](./screenshots/image11.png)


### 2.6 — Building a formatted, conditional summary table

I built a **Customizable Table** summarizing total fare amount, total trip distance, and average fare amount by day of week — formatting the currency columns with a `$` prefix and applying **conditional font coloring** (blue by default, green when `total_fare_amount` exceeds 30,000) so the table itself flags which days were exceptional, without needing a separate chart.

![Customized summary table with conditional formatting](./screenshots/image12.png)


### 2.7 — A searchable, sortable detail table

For a drill-down use case, I built a second table (day of week, trip distance, fare amount) with a **searchable** `day_of_week` column, filtered to Sunday only, and sorted by fare amount descending — turning a static table into a lightweight interactive lookup tool.

![Searchable and sortable detail table filtered to Sunday](./screenshots/image13.png)

---


## Chapter 3 — Building, Editing, and Operating Dashboards

This chapter was about treating a dashboard as a living product: assembling visualizations into a coherent layout, editing it after the fact, publishing it, and keeping it fresh with scheduled refreshes and filter widgets.


### 3.1 — Assembling a dashboard from scratch, with a parameter

I built a **"Trips"** dashboard combining a scatter plot ("Daily Fare Trends": `trip_distance` vs `fare_amount`) with a counter widget ("Total Trips"), styled the scatter points red, and replaced the hard-coded `trip_distance < 10` filter with a `:max_distance` **parameter** set to 20 — making the dashboard's scope adjustable without editing SQL.

**Result:** with the parameter applied, the dashboard showed **20,694 total trips**.

![Trips dashboard with scatter plot, counter widget, and a distance parameter](./screenshots/image14.png)


### 3.2 — Extending an existing dashboard with cross-filtering

Rather than rebuilding from scratch, I imported an existing dashboard and added a new bar chart (total fare by weekday) directly onto the canvas, then used Databricks' **cross-filtering** to drill into a single day from the dashboard itself.

**Result:** cross-filtering to **Saturday** showed a total trip count of **3,255**.

![Adding a new visualization and cross-filtering by weekday](./screenshots/image15.png)


### 3.3 — Editing an existing dashboard's layout

Dashboards evolve. I practiced the maintenance side: removing two widgets that were no longer needed ("Total Trips" and "Route Revenue Attribution"), repositioning and resizing the "Daily Fare Trends" widget to span the full width, and relabeling its axes to "Trip Distance (miles)" and "Fare Amount (USD)" for clarity.

![Dashboard after removing widgets and repositioning the remaining chart](./screenshots/image16.png)

![Resized and relabeled Daily Fare Trends widget](./screenshots/image17.png)


### 3.4 — Publishing and manually refreshing a dashboard

A dashboard only helps stakeholders once it's published. I published a dashboard with default settings, switched it to the **published view**, and manually triggered a refresh to confirm it reflects current data.

![Publishing the dashboard](./screenshots/image18.png)

![Manually refreshing the published dashboard](./screenshots/image19.png)


### 3.5 — Scheduling automatic refreshes

To avoid relying on someone remembering to refresh a dashboard manually, I set up a scheduled refresh (initially daily at 9 PM UTC, then edited to 5 AM UTC) — the kind of small operational detail that keeps a dashboard trustworthy over time.

**Result:** the published dashboard flagged **4 routes with total revenue of zero**, a data-quality signal worth investigating.

![Setting a daily scheduled refresh](./screenshots/image20.png)

![Editing the schedule to a new refresh time](./screenshots/image21.png)


### 3.6 — Interactive filter widgets

Finally, I added a **day-of-week filter widget** to a dashboard and narrowed it to Monday trips picked up in zip code 10007, demonstrating how filter widgets let end users self-serve their own view of the data without needing to write SQL.

![Adding a day-of-week filter widget](./screenshots/image22.png)

![Dashboard filtered to Monday, pickup zip 10007](./screenshots/image23.png)

---


## Chapter 4 — Dashboard Lifecycle Management

The final chapter covered what happens *around* a dashboard once it exists: safely iterating on a copy, sharing it externally, deleting it responsibly, managing permissions for different audiences, and alerting on the data itself.


### 4.1 — Cloning a dashboard for a different audience

The marketing team wanted a version of the taxi dashboard focused on long-distance trips, without touching the original the operations team relies on. I **cloned** the dashboard ("NYC Taxi Performance - Marketing") and updated its filters to `trip_distance > 10`, `0 < fare_amount < 100`.

**Result:** the long-distance-focused clone showed **1,118 total trips** — a much smaller, more targeted segment than the full dataset, exactly as expected.

![Cloned dashboard filtered to long-distance trips](./screenshots/image24.png)


### 4.2 — Exporting a dashboard for offline sharing

For stakeholders without Databricks workspace access, I exported a draft-view dashboard so it could be shared as a standalone file — Databricks exports dashboards as **JSON**, which can be re-imported elsewhere (as I did throughout this chapter when importing `*_exercise` dashboards).

### 4.3 — Deleting a dashboard responsibly

For a dashboard tied to a marketing campaign that had concluded, I moved it to trash and then permanently deleted it — while confirming the safety net: an accidentally-trashed dashboard can be recovered from the **Trash** section before permanent deletion.

### 4.4 — Sharing within the workspace with tiered permissions

For a dashboard the whole operations team depends on, I set up **tiered sharing**: view-only access for all workspace users, edit access for a specific group of students, and manage access for Admins — and reasoned through the standard troubleshooting checklist when a team member reports they can't see it (check individual permissions, check folder-level permissions, ask them to refresh, and escalate to a workspace admin if the issue persists).

![Configuring tiered sharing permissions on a workspace dashboard](./screenshots/image25.png)


### 4.5 — Sharing with account members outside the workspace

For teammates in the same Databricks account but outside this specific workspace, I published the dashboard and shared it account-wide — and confirmed an important nuance: **edits made in draft view are invisible to other users until the dashboard is explicitly republished**, which matters for anyone coordinating changes across a team.

### 4.6 — Setting up a data-quality alert

Finally, I created an **Alert** ("low fare warning") on a saved query, triggering when the minimum `fare_amount` drops below **$0.01** — a proxy for bad/erroneous fare data reaching the table — with a custom notification subject, running on a daily schedule.

**Result:** running the alert manually confirmed it would trigger — meaning the underlying data does contain at least one near-zero fare that the team should investigate.

![Configuring the low-fare data-quality alert](./screenshots/image26.png)

![Alert trigger conditions and notification template](./screenshots/image27.png)

---


## ✅ Skills demonstrated in this module

- Choosing the appropriate chart type (bar, line, combo, pie, histogram, choropleth map, marker map) for a given analytical question.
- Applying formatting best practices: axis labeling, color for emphasis, precise data-label formatting, and conditional table formatting.
- Assembling dashboards from multiple visualizations, including counters, scatter plots, and cross-filtering.
- Full dashboard lifecycle management: editing, publishing, scheduled refreshing, cloning, exporting, deleting, and tiered permission sharing.
- Setting up data-quality alerts on top of a saved SQL query.
